# Kaggle Phase 2 — Model & Component Definitions + Smoke Test

Second of 3 notebooks. Defines every reusable component (variable-length residual GRU,
on-the-fly window dataset, training utilities, baselines, and the drift-aware components
from the proposal: DriftMonitor, AdaptiveThreshold, OnlineAdapter), smoke-tests them, and
exports a single `model_defs.py` for `kagglephase3` to import — no duplicated code between
notebooks.

**Inputs (optional):** the `kagglephase1-output` dataset via Add Input — if attached, the
smoke test also runs against the real Phase-1 files; if not, it runs on synthetic data only.

**Output:** `model_defs.py` (+ `kagglephase2_defs.zip`). Download from the Output tab and
upload as a Kaggle Dataset named e.g. `kagglephase2-defs`, then attach to `kagglephase3`.


## Step 1: Imports + Optional Phase-1 Input Detection

In [8]:
import os, gc, json, time, random, inspect
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_KAGGLE = os.path.exists('/kaggle')
print(f"PyTorch {torch.__version__} | device: {device}")

phase1_dir = None
if IN_KAGGLE and Path('/kaggle/input').exists():
    hits = sorted(Path('/kaggle/input').glob('**/features_injected.npy'), key=lambda p: len(p.parts))
    if hits:
        phase1_dir = hits[0].parent
        print(f"Phase-1 output found: {phase1_dir} -- smoke test will also use real files")
if phase1_dir is None:
    print("Phase-1 output NOT attached -- smoke test will use synthetic data only.")
    print("(To include the real-file check: Add Input -> your 'kagglephase1-output' dataset.)")


PyTorch 2.10.0+cu128 | device: cuda
Phase-1 output found: /kaggle/input/datasets/thanakaran/kagglephase1-output -- smoke test will also use real files


## Step 2: AdaptiveGRUModel (variable-length, packed, residual-anchored)

In [9]:
def initialize_weights(module):
    if isinstance(module, nn.GRU):
        for name, param in module.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    elif isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        nn.init.zeros_(module.bias)


class AdaptiveGRUModel(nn.Module):
    """Stacked GRU for variable-length windows (500-1000 steps, packed sequences).

    residual_indices: indices of the target columns within the input features.
    When set, forward() returns last_observed_value + correction, with the
    correction head zero-initialized so the model starts exactly at persistence.
    """

    def __init__(self, input_size=27, hidden_size=128, num_layers=2,
                 output_size=4, dropout=0.2, residual_indices=None):
        super().__init__()
        self.input_size, self.hidden_size = input_size, hidden_size
        self.num_layers, self.output_size = num_layers, output_size
        self.residual_indices = residual_indices
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, output_size)
        self.apply(initialize_weights)
        if residual_indices is not None:
            self.register_buffer('_res_idx', torch.tensor(residual_indices, dtype=torch.long),
                                 persistent=False)
            nn.init.zeros_(self.fc2.weight)
            nn.init.zeros_(self.fc2.bias)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        out = self.fc2(self.relu(self.fc1(self.dropout(h_n[-1]))))
        if self.residual_indices is not None:
            last = x[torch.arange(x.size(0), device=x.device), lengths - 1]
            out = out + last[:, self._res_idx]
        return out

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


## Step 3: WindowDataset + Padding Collate (on-the-fly, memmap-backed)

In [10]:
class WindowDataset(Dataset):
    """Materializes windows per item from a memmapped (n_rows, n_feat) array.

    windows: (N, 2) int32 [anchor_end_exclusive, lookback_length].
    X = features[end-L:end]; y = features[end-1+horizon, target_idx].
    """

    def __init__(self, features_path, windows, horizon, target_idx):
        self.feat = np.load(features_path, mmap_mode='r')
        self.windows = windows
        self.h = int(horizon)
        self.tidx = np.asarray(target_idx)

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, i):
        end, L = int(self.windows[i, 0]), int(self.windows[i, 1])
        x = torch.from_numpy(np.array(self.feat[end - L:end], dtype=np.float32))
        y = torch.from_numpy(np.array(self.feat[end - 1 + self.h, self.tidx], dtype=np.float32))
        return x, y, L


def collate_pad(batch):
    xs, ys, Ls = zip(*batch)
    T = max(Ls)
    X = torch.zeros(len(xs), T, xs[0].shape[1], dtype=torch.float32)
    for i, x in enumerate(xs):
        X[i, :x.shape[0]] = x
    return X, torch.stack(ys), torch.tensor(Ls, dtype=torch.long)


def make_loader(features_path, windows, horizon, target_idx, batch_size=64, shuffle=False):
    ds = WindowDataset(features_path, windows, horizon, target_idx)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      collate_fn=collate_pad, pin_memory=torch.cuda.is_available())


## Step 4: Training Utilities

In [11]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.0):
        self.patience, self.min_delta = patience, min_delta
        self.best = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best - self.min_delta:
            self.best = val_loss
            self.counter = 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True
        return False


def train_one_epoch(model, loader, optimizer, criterion, dev, grad_clip=1.0):
    model.train()
    total = 0.0
    for X, y, L in loader:
        X, y = X.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(X, L), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()
        total += loss.item() * len(X)
    return total / len(loader.dataset)


def compute_metrics(preds_norm, targets_norm, target_std, target_mean, target_names):
    preds_real = preds_norm * target_std + target_mean
    targets_real = targets_norm * target_std + target_mean
    abs_err = np.abs(preds_real - targets_real)
    eps = 1e-6
    near0 = np.abs(targets_real) < eps
    ape = np.where(near0, np.nan, abs_err / np.where(near0, eps, np.abs(targets_real)) * 100.0)
    out = {}
    for i, name in enumerate(target_names):
        out[name] = {'mae': float(abs_err[:, i].mean()),
                     'rmse': float(np.sqrt(((preds_real[:, i] - targets_real[:, i]) ** 2).mean())),
                     'mape': float(np.nanmean(ape[:, i]))}
    out['mape_mean'] = float(np.nanmean([out[n]['mape'] for n in target_names]))
    return out


@torch.no_grad()
def predict_all(model, loader, dev):
    model.eval()
    preds, tgts = [], []
    for X, y, L in loader:
        preds.append(model(X.to(dev, non_blocking=True), L).cpu())
        tgts.append(y)
    return torch.cat(preds).numpy(), torch.cat(tgts).numpy()


def evaluate(model, loader, dev, target_std, target_mean, target_names):
    p, t = predict_all(model, loader, dev)
    m = compute_metrics(p, t, target_std, target_mean, target_names)
    m['loss'] = float(((p - t) ** 2).mean())
    return m


def train_model(model, train_loader, val_loader, dev, ckpt_path, epochs=30, lr=1e-3,
                patience=7, scheduler_patience=4, grad_clip=1.0, log_every=5):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=scheduler_patience, min_lr=1e-6)
    stopper = EarlyStopping(patience=patience)
    best = float('inf')
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, criterion, dev, grad_clip)
        with torch.no_grad():
            model.eval()
            v_tot = 0.0
            n = 0
            for X, y, L in val_loader:
                X, y = X.to(dev), y.to(dev)
                v_tot += nn.functional.mse_loss(model(X, L), y, reduction='sum').item()
                n += y.numel()
            vl = v_tot / n
        scheduler.step(vl)
        if stopper(vl):
            best = vl
            torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch,
                        'val_loss': vl}, ckpt_path)
        if epoch == 1 or epoch % log_every == 0:
            print(f"  epoch {epoch:3d} | train={tr:.6f} | val={vl:.6f} | "
                  f"patience={stopper.counter}/{patience} | {time.time()-t0:.0f}s")
        if stopper.early_stop:
            print(f"  early stop at epoch {epoch}")
            break
    print(f"  done: best_val={best:.6f} | {time.time()-t0:.0f}s")
    return best


## Step 5: Baselines + Drift-Aware Components (proposal items)

- `persistence_preds` / `ses_preds` — the two non-learned baselines.
- `DriftMonitor` — moving-average error + z-score statistical check (proposal: "error
  monitoring using moving average and statistical checks").
- `AdaptiveThreshold` — rolling error-percentile confidence band (proposal: "adaptive
  thresholding to dynamically adjust prediction confidence").
- `OnlineAdapter` — error-triggered incremental fine-tuning on recent already-seen
  windows (proposal: "online/incremental learning with error-triggered updates").


In [12]:
def persistence_preds(features, windows, target_idx):
    anchors = windows[:, 0] - 1
    return np.asarray(features[anchors][:, target_idx], dtype=np.float64)


def ses_fit_alpha(features, windows, horizon, target_idx, tail=240, max_n=20000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(windows), size=min(max_n, len(windows)), replace=False)
    alphas = np.round(np.arange(0.05, 1.0001, 0.05), 2)
    fitted = []
    for ti, col in enumerate(target_idx):
        segs = np.stack([np.array(features[max(e - tail, e - L):e, col], dtype=np.float64)[-tail:]
                         for e, L in windows[idx]])
        tgt = np.array([features[e - 1 + horizon, col] for e, L in windows[idx]], dtype=np.float64)
        best_a, best_err = 1.0, np.inf
        for a in alphas:
            w = a * (1.0 - a) ** np.arange(segs.shape[1] - 1, -1, -1)
            w /= w.sum()
            err = float(np.mean((segs @ w - tgt) ** 2))
            if err < best_err:
                best_a, best_err = a, err
        fitted.append(best_a)
    return fitted


def ses_preds(features, windows, target_idx, alphas, tail=240):
    out = np.empty((len(windows), len(target_idx)), dtype=np.float64)
    for ti, (col, a) in enumerate(zip(target_idx, alphas)):
        w = a * (1.0 - a) ** np.arange(tail - 1, -1, -1)
        w /= w.sum()
        segs = np.stack([np.array(features[e - tail:e, col], dtype=np.float64) for e, L in windows])
        out[:, ti] = segs @ w
    return out


class DriftMonitor:
    """EWMA of chunk-level error + z-score vs a frozen reference period."""

    def __init__(self, ewma_alpha=0.3, z_threshold=3.0, warmup_chunks=8, sustain=2):
        self.alpha, self.z, self.warmup, self.sustain = ewma_alpha, z_threshold, warmup_chunks, sustain
        self.ref = []
        self.ewma = None
        self.hits = 0
        self.drifting = False

    def update(self, chunk_error):
        self.ewma = chunk_error if self.ewma is None else             self.alpha * chunk_error + (1 - self.alpha) * self.ewma
        if len(self.ref) < self.warmup:
            self.ref.append(chunk_error)
            return False
        mu = float(np.mean(self.ref))
        sd = float(np.std(self.ref)) or 1e-12
        if (self.ewma - mu) / sd > self.z:
            self.hits += 1
        else:
            self.hits = 0
        self.drifting = self.hits >= self.sustain
        return self.drifting


class AdaptiveThreshold:
    """Confidence band from rolling error percentiles over recent chunks."""

    def __init__(self, window_chunks=20, lo_pct=50, hi_pct=90):
        self.window, self.lo, self.hi = window_chunks, lo_pct, hi_pct
        self.buf = []

    def update(self, chunk_abs_errors):
        self.buf.append(np.asarray(chunk_abs_errors))
        if len(self.buf) > self.window:
            self.buf.pop(0)
        allv = np.concatenate(self.buf)
        return float(np.percentile(allv, self.lo)), float(np.percentile(allv, self.hi))


class OnlineAdapter:
    """Error-triggered incremental fine-tune on the most recent seen windows."""

    def __init__(self, features_path, target_idx, horizon, lr=1e-4, recent=4096,
                 batch_size=64, epochs=1, grad_clip=1.0):
        self.features_path, self.tidx, self.h = features_path, target_idx, horizon
        self.lr, self.recent, self.bs, self.epochs, self.clip = lr, recent, batch_size, epochs, grad_clip
        self.n_updates = 0

    def adapt(self, model, seen_windows, dev):
        recent = seen_windows[-self.recent:]
        loader = make_loader(self.features_path, recent, self.h, self.tidx,
                             batch_size=self.bs, shuffle=True)
        optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()
        model.train()
        for _ in range(self.epochs):
            for X, y, L in loader:
                X, y = X.to(dev), y.to(dev)
                optimizer.zero_grad()
                loss = criterion(model(X, L), y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=self.clip)
                optimizer.step()
        model.eval()
        self.n_updates += 1
        return model


## Step 6: Smoke Test (synthetic always; real Phase-1 files if attached)

In [13]:
import tempfile

set_seed(42)
tmp = Path(tempfile.mkdtemp())
n_rows, n_feat = 4000, 27
tidx = [1, 5, 6, 4]
rng = np.random.default_rng(0)
feat = rng.normal(0, 1, (n_rows, n_feat)).astype(np.float32)
for c in tidx:
    feat[:, c] = np.cumsum(rng.normal(0, 0.05, n_rows)).astype(np.float32)
np.save(tmp / 'feat.npy', feat)
wins = np.array([[e, int(rng.integers(500, 1001))] for e in range(1005, n_rows - 10, 7)], dtype=np.int32)

model = AdaptiveGRUModel(residual_indices=tidx).to(device)
print(f"[1] model: {model.n_params():,} params")
loader = make_loader(tmp / 'feat.npy', wins[:128], horizon=1, target_idx=tidx, batch_size=32)
X, y, L = next(iter(loader))
out = model(X.to(device), L)
assert out.shape == (len(X), 4), out.shape
print(f"[1] forward: X={tuple(X.shape)} lengths {L.min().item()}-{L.max().item()} -> out={tuple(out.shape)}  PASS")

anchors = X[torch.arange(len(X)), L - 1][:, tidx].to(device)
assert torch.allclose(out, anchors, atol=1e-6), "zero-init residual must equal persistence at init"
print("[2] residual identity at init (output == last observed value)  PASS")

crit = nn.MSELoss()
loss = crit(model(X.to(device), L), y.to(device))
loss.backward()
grads = [p.grad for p in model.parameters() if p.requires_grad]
assert all(g is not None for g in grads) and all(torch.isfinite(g).all() for g in grads)
model.zero_grad()
print(f"[3] backward pass: loss={loss.item():.6f}, all grads finite  PASS")

p = persistence_preds(feat, wins[:200], tidx)
assert p.shape == (200, 4)
alphas = ses_fit_alpha(feat, wins, horizon=1, target_idx=tidx, max_n=500)
s = ses_preds(feat, wins[:200], tidx, alphas)
print(f"[4] baselines: persistence {p.shape}, SES alphas={alphas}  PASS")

dm = DriftMonitor(warmup_chunks=4, sustain=2)
fired_at = None
for i, e in enumerate([1.0, 1.1, 0.9, 1.0, 1.0, 1.05, 8.0, 9.0, 9.5]):
    if dm.update(e) and fired_at is None:
        fired_at = i
assert fired_at is not None, "drift monitor never fired on a 8x error jump"
at = AdaptiveThreshold(window_chunks=5)
lo, hi = at.update(np.abs(rng.normal(0, 1, 500)))
assert lo < hi
print(f"[5] DriftMonitor fired at chunk {fired_at} on sustained error jump; "
      f"AdaptiveThreshold band=({lo:.2f},{hi:.2f})  PASS")

oa = OnlineAdapter(tmp / 'feat.npy', tidx, horizon=1, recent=256, epochs=1)
before = [p.detach().clone() for p in model.parameters()]
oa.adapt(model, wins[:512], device)
changed = any(not torch.equal(b, p.detach()) for b, p in zip(before, model.parameters()))
assert changed and oa.n_updates == 1
print("[6] OnlineAdapter fine-tune updates weights  PASS")

if phase1_dir is not None:
    w_real = np.load(phase1_dir / 'windows_val.npy')[:64]
    fc = json.load(open(phase1_dir / 'feature_cols.json'))
    rl = make_loader(phase1_dir / 'features_injected.npy', w_real, 1, fc['target_idx'], batch_size=16)
    Xr, yr, Lr = next(iter(rl))
    outr = model(Xr.to(device), Lr)
    assert outr.shape == (len(Xr), 4)
    print(f"[7] REAL Phase-1 files: batch X={tuple(Xr.shape)} -> out={tuple(outr.shape)}  PASS")
else:
    print("[7] real-file check skipped (phase1 dataset not attached)")

print()
print("ALL SMOKE TESTS PASSED")


[1] model: 167,876 params
[1] forward: X=(32, 981, 27) lengths 505-981 -> out=(32, 4)  PASS
[2] residual identity at init (output == last observed value)  PASS
[3] backward pass: loss=0.002762, all grads finite  PASS
[4] baselines: persistence (200, 4), SES alphas=[np.float64(0.95), np.float64(0.95), np.float64(1.0), np.float64(1.0)]  PASS
[5] DriftMonitor fired at chunk 7 on sustained error jump; AdaptiveThreshold band=(0.70,1.78)  PASS
[6] OnlineAdapter fine-tune updates weights  PASS
[7] REAL Phase-1 files: batch X=(16, 1000, 27) -> out=(16, 4)  PASS

ALL SMOKE TESTS PASSED


## Step 7: Export `model_defs.py`

Does NOT use `inspect.getsource()` -- Kaggle's kernel has no `__main__.__file__`,
which makes `inspect.getsource()` raise `OSError: source code not available` for
any class (functions are usually fine via linecache, but classes are not, in this
specific environment). Instead the Step 2/3/4/5 definition cells' source is embedded
here as literal (json-escaped) strings -- zero dependency on runtime introspection
or the notebook file being discoverable on disk, so this works in any environment.


In [14]:
import ast, zipfile

HEADER = "\"\"\"Auto-exported by kagglephase2.ipynb -- shared definitions for kagglephase3.\"\"\"\nimport time, random\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom torch.nn.utils.rnn import pack_padded_sequence\n\n"

# Embedded verbatim (not via inspect.getsource -- Kaggle's kernel has no
# __main__.__file__, which makes inspect.getsource() raise OSError for any
# class in this environment; functions are usually fine via linecache, classes
# are not). These are the exact contents of the Step 2/3/4/5 definition cells
# above, duplicated here as string literals (json-escaped, not hand-escaped)
# so export has zero dependency on runtime introspection or the notebook file
# being discoverable on disk.
DEF_BLOCKS = [
"def initialize_weights(module):\n    if isinstance(module, nn.GRU):\n        for name, param in module.named_parameters():\n            if 'weight_ih' in name:\n                nn.init.xavier_uniform_(param)\n            elif 'weight_hh' in name:\n                nn.init.orthogonal_(param)\n            elif 'bias' in name:\n                nn.init.zeros_(param)\n    elif isinstance(module, nn.Linear):\n        nn.init.xavier_uniform_(module.weight)\n        nn.init.zeros_(module.bias)\n\n\nclass AdaptiveGRUModel(nn.Module):\n    \"\"\"Stacked GRU for variable-length windows (500-1000 steps, packed sequences).\n\n    residual_indices: indices of the target columns within the input features.\n    When set, forward() returns last_observed_value + correction, with the\n    correction head zero-initialized so the model starts exactly at persistence.\n    \"\"\"\n\n    def __init__(self, input_size=27, hidden_size=128, num_layers=2,\n                 output_size=4, dropout=0.2, residual_indices=None):\n        super().__init__()\n        self.input_size, self.hidden_size = input_size, hidden_size\n        self.num_layers, self.output_size = num_layers, output_size\n        self.residual_indices = residual_indices\n        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,\n                          dropout=dropout if num_layers > 1 else 0.0)\n        self.dropout = nn.Dropout(dropout)\n        self.fc1 = nn.Linear(hidden_size, 64)\n        self.relu = nn.ReLU()\n        self.fc2 = nn.Linear(64, output_size)\n        self.apply(initialize_weights)\n        if residual_indices is not None:\n            self.register_buffer('_res_idx', torch.tensor(residual_indices, dtype=torch.long),\n                                 persistent=False)\n            nn.init.zeros_(self.fc2.weight)\n            nn.init.zeros_(self.fc2.bias)\n\n    def forward(self, x, lengths):\n        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)\n        _, h_n = self.gru(packed)\n        out = self.fc2(self.relu(self.fc1(self.dropout(h_n[-1]))))\n        if self.residual_indices is not None:\n            last = x[torch.arange(x.size(0), device=x.device), lengths - 1]\n            out = out + last[:, self._res_idx]\n        return out\n\n    def n_params(self):\n        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n",
"class WindowDataset(Dataset):\n    \"\"\"Materializes windows per item from a memmapped (n_rows, n_feat) array.\n\n    windows: (N, 2) int32 [anchor_end_exclusive, lookback_length].\n    X = features[end-L:end]; y = features[end-1+horizon, target_idx].\n    \"\"\"\n\n    def __init__(self, features_path, windows, horizon, target_idx):\n        self.feat = np.load(features_path, mmap_mode='r')\n        self.windows = windows\n        self.h = int(horizon)\n        self.tidx = np.asarray(target_idx)\n\n    def __len__(self):\n        return len(self.windows)\n\n    def __getitem__(self, i):\n        end, L = int(self.windows[i, 0]), int(self.windows[i, 1])\n        x = torch.from_numpy(np.array(self.feat[end - L:end], dtype=np.float32))\n        y = torch.from_numpy(np.array(self.feat[end - 1 + self.h, self.tidx], dtype=np.float32))\n        return x, y, L\n\n\ndef collate_pad(batch):\n    xs, ys, Ls = zip(*batch)\n    T = max(Ls)\n    X = torch.zeros(len(xs), T, xs[0].shape[1], dtype=torch.float32)\n    for i, x in enumerate(xs):\n        X[i, :x.shape[0]] = x\n    return X, torch.stack(ys), torch.tensor(Ls, dtype=torch.long)\n\n\ndef make_loader(features_path, windows, horizon, target_idx, batch_size=64, shuffle=False):\n    ds = WindowDataset(features_path, windows, horizon, target_idx)\n    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0,\n                      collate_fn=collate_pad, pin_memory=torch.cuda.is_available())\n",
"def set_seed(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\nclass EarlyStopping:\n    def __init__(self, patience=7, min_delta=0.0):\n        self.patience, self.min_delta = patience, min_delta\n        self.best = float('inf')\n        self.counter = 0\n        self.early_stop = False\n\n    def __call__(self, val_loss):\n        if val_loss < self.best - self.min_delta:\n            self.best = val_loss\n            self.counter = 0\n            return True\n        self.counter += 1\n        if self.counter >= self.patience:\n            self.early_stop = True\n        return False\n\n\ndef train_one_epoch(model, loader, optimizer, criterion, dev, grad_clip=1.0):\n    model.train()\n    total = 0.0\n    for X, y, L in loader:\n        X, y = X.to(dev, non_blocking=True), y.to(dev, non_blocking=True)\n        optimizer.zero_grad()\n        loss = criterion(model(X, L), y)\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)\n        optimizer.step()\n        total += loss.item() * len(X)\n    return total / len(loader.dataset)\n\n\ndef compute_metrics(preds_norm, targets_norm, target_std, target_mean, target_names):\n    preds_real = preds_norm * target_std + target_mean\n    targets_real = targets_norm * target_std + target_mean\n    abs_err = np.abs(preds_real - targets_real)\n    eps = 1e-6\n    near0 = np.abs(targets_real) < eps\n    ape = np.where(near0, np.nan, abs_err / np.where(near0, eps, np.abs(targets_real)) * 100.0)\n    out = {}\n    for i, name in enumerate(target_names):\n        out[name] = {'mae': float(abs_err[:, i].mean()),\n                     'rmse': float(np.sqrt(((preds_real[:, i] - targets_real[:, i]) ** 2).mean())),\n                     'mape': float(np.nanmean(ape[:, i]))}\n    out['mape_mean'] = float(np.nanmean([out[n]['mape'] for n in target_names]))\n    return out\n\n\n@torch.no_grad()\ndef predict_all(model, loader, dev):\n    model.eval()\n    preds, tgts = [], []\n    for X, y, L in loader:\n        preds.append(model(X.to(dev, non_blocking=True), L).cpu())\n        tgts.append(y)\n    return torch.cat(preds).numpy(), torch.cat(tgts).numpy()\n\n\ndef evaluate(model, loader, dev, target_std, target_mean, target_names):\n    p, t = predict_all(model, loader, dev)\n    m = compute_metrics(p, t, target_std, target_mean, target_names)\n    m['loss'] = float(((p - t) ** 2).mean())\n    return m\n\n\ndef train_model(model, train_loader, val_loader, dev, ckpt_path, epochs=30, lr=1e-3,\n                patience=7, scheduler_patience=4, grad_clip=1.0, log_every=5):\n    criterion = nn.MSELoss()\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr)\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n        optimizer, mode='min', factor=0.5, patience=scheduler_patience, min_lr=1e-6)\n    stopper = EarlyStopping(patience=patience)\n    best = float('inf')\n    t0 = time.time()\n    for epoch in range(1, epochs + 1):\n        tr = train_one_epoch(model, train_loader, optimizer, criterion, dev, grad_clip)\n        with torch.no_grad():\n            model.eval()\n            v_tot = 0.0\n            n = 0\n            for X, y, L in val_loader:\n                X, y = X.to(dev), y.to(dev)\n                v_tot += nn.functional.mse_loss(model(X, L), y, reduction='sum').item()\n                n += y.numel()\n            vl = v_tot / n\n        scheduler.step(vl)\n        if stopper(vl):\n            best = vl\n            torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch,\n                        'val_loss': vl}, ckpt_path)\n        if epoch == 1 or epoch % log_every == 0:\n            print(f\"  epoch {epoch:3d} | train={tr:.6f} | val={vl:.6f} | \"\n                  f\"patience={stopper.counter}/{patience} | {time.time()-t0:.0f}s\")\n        if stopper.early_stop:\n            print(f\"  early stop at epoch {epoch}\")\n            break\n    print(f\"  done: best_val={best:.6f} | {time.time()-t0:.0f}s\")\n    return best\n",
"def persistence_preds(features, windows, target_idx):\n    anchors = windows[:, 0] - 1\n    return np.asarray(features[anchors][:, target_idx], dtype=np.float64)\n\n\ndef ses_fit_alpha(features, windows, horizon, target_idx, tail=240, max_n=20000, seed=0):\n    rng = np.random.default_rng(seed)\n    idx = rng.choice(len(windows), size=min(max_n, len(windows)), replace=False)\n    alphas = np.round(np.arange(0.05, 1.0001, 0.05), 2)\n    fitted = []\n    for ti, col in enumerate(target_idx):\n        segs = np.stack([np.array(features[max(e - tail, e - L):e, col], dtype=np.float64)[-tail:]\n                         for e, L in windows[idx]])\n        tgt = np.array([features[e - 1 + horizon, col] for e, L in windows[idx]], dtype=np.float64)\n        best_a, best_err = 1.0, np.inf\n        for a in alphas:\n            w = a * (1.0 - a) ** np.arange(segs.shape[1] - 1, -1, -1)\n            w /= w.sum()\n            err = float(np.mean((segs @ w - tgt) ** 2))\n            if err < best_err:\n                best_a, best_err = a, err\n        fitted.append(best_a)\n    return fitted\n\n\ndef ses_preds(features, windows, target_idx, alphas, tail=240):\n    out = np.empty((len(windows), len(target_idx)), dtype=np.float64)\n    for ti, (col, a) in enumerate(zip(target_idx, alphas)):\n        w = a * (1.0 - a) ** np.arange(tail - 1, -1, -1)\n        w /= w.sum()\n        segs = np.stack([np.array(features[e - tail:e, col], dtype=np.float64) for e, L in windows])\n        out[:, ti] = segs @ w\n    return out\n\n\nclass DriftMonitor:\n    \"\"\"EWMA of chunk-level error + z-score vs a frozen reference period.\"\"\"\n\n    def __init__(self, ewma_alpha=0.3, z_threshold=3.0, warmup_chunks=8, sustain=2):\n        self.alpha, self.z, self.warmup, self.sustain = ewma_alpha, z_threshold, warmup_chunks, sustain\n        self.ref = []\n        self.ewma = None\n        self.hits = 0\n        self.drifting = False\n\n    def update(self, chunk_error):\n        self.ewma = chunk_error if self.ewma is None else             self.alpha * chunk_error + (1 - self.alpha) * self.ewma\n        if len(self.ref) < self.warmup:\n            self.ref.append(chunk_error)\n            return False\n        mu = float(np.mean(self.ref))\n        sd = float(np.std(self.ref)) or 1e-12\n        if (self.ewma - mu) / sd > self.z:\n            self.hits += 1\n        else:\n            self.hits = 0\n        self.drifting = self.hits >= self.sustain\n        return self.drifting\n\n\nclass AdaptiveThreshold:\n    \"\"\"Confidence band from rolling error percentiles over recent chunks.\"\"\"\n\n    def __init__(self, window_chunks=20, lo_pct=50, hi_pct=90):\n        self.window, self.lo, self.hi = window_chunks, lo_pct, hi_pct\n        self.buf = []\n\n    def update(self, chunk_abs_errors):\n        self.buf.append(np.asarray(chunk_abs_errors))\n        if len(self.buf) > self.window:\n            self.buf.pop(0)\n        allv = np.concatenate(self.buf)\n        return float(np.percentile(allv, self.lo)), float(np.percentile(allv, self.hi))\n\n\nclass OnlineAdapter:\n    \"\"\"Error-triggered incremental fine-tune on the most recent seen windows.\"\"\"\n\n    def __init__(self, features_path, target_idx, horizon, lr=1e-4, recent=4096,\n                 batch_size=64, epochs=1, grad_clip=1.0):\n        self.features_path, self.tidx, self.h = features_path, target_idx, horizon\n        self.lr, self.recent, self.bs, self.epochs, self.clip = lr, recent, batch_size, epochs, grad_clip\n        self.n_updates = 0\n\n    def adapt(self, model, seen_windows, dev):\n        recent = seen_windows[-self.recent:]\n        loader = make_loader(self.features_path, recent, self.h, self.tidx,\n                             batch_size=self.bs, shuffle=True)\n        optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)\n        criterion = nn.MSELoss()\n        model.train()\n        for _ in range(self.epochs):\n            for X, y, L in loader:\n                X, y = X.to(dev), y.to(dev)\n                optimizer.zero_grad()\n                loss = criterion(model(X, L), y)\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=self.clip)\n                optimizer.step()\n        model.eval()\n        self.n_updates += 1\n        return model\n"
]

src = HEADER + "\n\n".join(b.strip() for b in DEF_BLOCKS)
ast.parse(src)  # syntax check before writing anything
compile(src, "model_defs.py", "exec")

out_py = Path("/kaggle/working/model_defs.py") if IN_KAGGLE else Path("./model_defs.py")
out_py.write_text(src, encoding="utf-8")
print(f"Wrote {out_py} ({out_py.stat().st_size/1024:.1f} KB) -- syntax verified, "
      f"{len(DEF_BLOCKS)} definition blocks embedded")

zip_path = out_py.parent / "kagglephase2_defs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(out_py, out_py.name)
print(f"Archive: {zip_path}")
print()
print("NEXT: Output tab -> download model_defs.py (or the zip) -> upload as a Kaggle")
print("Dataset named 'kagglephase2-defs' -> attach to kagglephase3 via Add Input.")


Wrote /kaggle/working/model_defs.py (12.0 KB) -- syntax verified, 4 definition blocks embedded
Archive: /kaggle/working/kagglephase2_defs.zip

NEXT: Output tab -> download model_defs.py (or the zip) -> upload as a Kaggle
Dataset named 'kagglephase2-defs' -> attach to kagglephase3 via Add Input.
